# Frozen Lake(얼어붙은 호수) · 지능 에이전트 관찰하기 (교과서 175쪽 보충)

지능 에이전트가 **미끄러운 얼음 위를 건너 목적지에 무사히 도착하는 경로**를 찾는 문제예요.

- **환경**: 4×4 칸으로 된 얼어붙은 호수 — 출발점(S, 왼쪽 위), 얼음(F, 안전한 곳), 구멍(H, 빠지면 게임 끝), 목적지(G, 오른쪽 아래)
- **지능 에이전트**: 호수를 건너는 사람(플레이어)
- **행동**: 한 번에 한 칸씩 왼쪽 · 아래 · 오른쪽 · 위 중 하나로 움직여요. 얼음이 미끄러워서 **원하는 방향으로 가지 않고 옆으로 미끄러질 때가 있어요.**
- **목표**: 출발점에서 목적지까지 구멍에 빠지지 않고 도착하기

위에서부터 셀을 하나씩 실행해요(셀 왼쪽의 ▶ 버튼 또는 Shift + Enter).

## 1. 라이브러리 설치하기
**gymnasium**은 Frozen Lake 같은 게임 환경을 제공하는 라이브러리예요. 설치에 몇십 초 걸려요.

In [ ]:
!pip -q install "gymnasium[toy-text]"

## 2. 라이브러리 불러오기

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

## 3. 게임 환경 만들기
`gym.make()`로 `FrozenLake-v1` 게임 환경을 만들어요. `is_slippery=True`는 얼음이 미끄럽다는 뜻이에요.

행동 번호: **0 왼쪽, 1 아래, 2 오른쪽, 3 위**

In [ ]:
env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=True, render_mode='rgb_array')
observation, info = env.reset(seed=0)

print('호수 지도 (S 출발, F 얼음, H 구멍, G 목적지)')
for row in env.unwrapped.desc:
    print(' '.join(c.decode() for c in row))
print('칸의 수:', env.observation_space.n, '/ 행동의 수:', env.action_space.n)
print('지금 에이전트가 있는 칸 번호:', observation)

plt.imshow(env.render())
plt.axis('off')
plt.show()

## 4. 무작위로 움직이는 에이전트 (교과서 예제)
에이전트가 `env.action_space.sample()`로 **행동을 무작위로 골라** 움직여요.

- `env.step(행동)`: 행동을 하면 환경이 바뀌고, 새 칸 번호 · 보상(목적지에 닿으면 1) · 게임이 끝났는지를 알려 줘요.
- `env.render()`: 지금 상태를 그림으로 그려요.
- 구멍에 빠지거나 목적지에 닿으면 `env.reset()`으로 다시 시작해요.

애니메이션을 보면서 에이전트가 **무엇을 인식하고, 어떻게 행동하는지** 관찰해요. (화면 위 글자: Episode = 몇 번째 판, Goal = 도착한 횟수)

In [ ]:
observation, info = env.reset()
episode, goal = 1, 0

for step in range(200):
    action = env.action_space.sample()          # 행동을 무작위로 고르기
    observation, reward, terminated, truncated, info = env.step(action)

    clear_output(wait=True)
    plt.imshow(env.render())
    plt.axis('off')
    plt.title(f'Episode {episode} | Goal {goal}')
    plt.show()
    time.sleep(0.05)

    if terminated or truncated:                 # 구멍에 빠졌거나 목적지에 닿았으면
        if reward == 1:
            goal += 1
        observation, info = env.reset()
        episode += 1

print(f'{episode}번째 판까지 해서 목적지에 {goal}번 도착했어요.')

## 5. 스스로 해결하기 · 무작위 에이전트는 몇 판에 한 번 도착할까?
4번 애니메이션은 판 수가 적어서 도착하는 모습을 보기 어려워요. **4번 코드를 조금 고쳐** 훨씬 많이 움직여 보세요.

**힌트**
- 4번 셀의 코드를 복사해 아래 셀에 붙여 넣어요.
- 그림을 그리는 줄(`clear_output`, `plt.`로 시작하는 줄)과 기다리는 줄(`time.sleep`)을 지워요. 그러면 아주 빨라져요.
- `range(200)`의 숫자를 크게(예: 3000) 바꿔 실행해요.
- 마지막에 나온 판 수와 도착 횟수를 보고, **100판에 몇 번꼴로 도착하는지** 어림해요.

In [ ]:
# 여기에 코드를 써요

## 6. 관찰 정리하기
무작위 에이전트는 아무렇게나 움직이므로 **학습이나 추론이 없어요.** 직접 건너 본 나(또는 잘 건너는 지능 에이전트)라면 단계마다 무엇을 할지 적고, 무작위 에이전트에게는 **없는 단계**도 「없음 — 그래서 ○○」처럼 적어요.
아래 칸을 **더블클릭**해서 적고, 코스웨어 화면의 메모 칸에도 같은 내용을 적어요.

- **인식**: 
- **학습**: 
- **추론**: 
- **행동**: 

## 7. 더 해 보기(선택) · 경험으로 배우는 에이전트
무작위 에이전트는 몇 판을 해도 나아지지 않아요. 이번에는 **경험에서 배우는 에이전트**를 만들어 견주어 봐요.

- 에이전트는 **점수표**(칸 16개 × 행동 4개)를 가지고 있어요. 처음에는 모두 0점이에요.
- 한 번 움직일 때마다, 그 칸에서 그 행동을 했더니 어떻게 되었는지(목적지에 가까워졌는지, 구멍에 빠졌는지)를 보고 점수표를 조금씩 고쳐요. → **학습**
- 다음에 같은 칸에 오면 점수가 가장 높은 행동을 골라요. 가끔은 일부러 다른 행동도 해 보며 새 길을 찾아요. → **추론**
- 이렇게 수천 판을 하면 점수표가 점점 좋아져요.

**코드를 다 이해하지 않아도 돼요.** 셀을 차례로 실행해서 무작위 에이전트와 결과만 견주어요. 아래 셀은 화면을 그리지 않고 5000판을 빠르게 연습해요(몇 초 걸려요).

In [ ]:
rng = np.random.default_rng(0)
table = np.zeros((env.observation_space.n, env.action_space.n))   # 점수표: 칸 × 행동

def choose(state, explore):
    if rng.random() < explore:                    # 가끔은 새로운 행동을 시도
        return int(rng.integers(env.action_space.n))
    best = np.flatnonzero(table[state] == table[state].max())
    return int(rng.choice(best))                  # 점수가 가장 높은 행동

episodes = 5000
for ep in range(episodes):
    state, info = env.reset()
    explore = max(0.05, 1 - ep / (episodes * 0.7))   # 처음엔 많이 시도, 갈수록 배운 대로
    done = False
    while not done:
        action = choose(state, explore)
        new_state, reward, terminated, truncated, info = env.step(action)
        # 점수표 고치기: 얻은 보상 + 다음 칸에서 기대할 수 있는 점수
        target = reward + 0.99 * table[new_state].max() * (not terminated)
        table[state, action] += 0.1 * (target - table[state, action])
        state = new_state
        done = terminated or truncated

print('연습 끝! 칸마다 가장 점수가 높은 행동:')
arrow = ['←', '↓', '→', '↑']
desc = env.unwrapped.desc
for r in range(4):
    line = ''
    for c in range(4):
        s = r * 4 + c
        cell = desc[r][c].decode()
        line += (cell if cell in 'HG' else arrow[int(table[s].argmax())]) + ' '
    print(line)

### 배운 에이전트는 100판 중 몇 번 도착할까?
5번에서 내가 어림한 **무작위 에이전트의 값과 견주어요.** 얼음이 미끄러워서 배운 에이전트도 서너 판에 한 판꼴로는 실패해요.

In [ ]:
goal = 0
for _ in range(100):
    state, info = env.reset()
    done = False
    while not done:
        action = int(table[state].argmax())     # 점수표에서 가장 점수가 높은 행동
        state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
    goal += int(reward == 1)

print('배운 에이전트: 100판 중', goal, '번 도착')

### 배운 에이전트가 건너는 모습 보기

In [ ]:
state, info = env.reset()
done, step = False, 0
while not done:
    action = int(table[state].argmax())
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    step += 1
    clear_output(wait=True)
    plt.imshow(env.render())
    plt.axis('off')
    plt.title(f'Step {step}')
    plt.show()
    time.sleep(0.2)

print('도착했어요!' if reward == 1 else '이번에는 미끄러져 실패했어요. 셀을 다시 실행해 봐요.')